In [15]:
import glob
import json
import os.path as osp
import openpyxl
import numpy as np
import pandas as pd

RESULTS_ROOT = osp.abspath(osp.join("..", "data", "results_mp_sampling"))
LOG_FILE_PATTERN = "raw_direct_query_responses_*.jsonl"
NPZ_FILENAME = "martingale_results.npz"
FLOOR_LOGPROB = np.log(1e-10)  # same floor model_calls.py uses for a label that never appears in top_logprobs

In [16]:
def _label_logprobs_from_top_logprobs(top_logprobs: list, label_chars: list) -> dict:
    """Combine token-spelling variants that map to the same class label (e.g.
    'A', ' A', 'a' all mean class A) by summing their probability mass in log
    space -- same logic as model_calls.py's _label_probs_from_top_logprobs,
    just kept in log space instead of immediately normalizing to a
    probability vector. Tokens that don't resolve to any of label_chars
    (e.g. 'F' when the question only has 5 choices) are ignored entirely, so
    they can never leak into a class's combined log-probability.

    Returns one combined log-probability per label; a label that never
    appears among the top candidates gets the same floor (log(1e-10)) that
    model_calls.py assigns before renormalizing.
    """
    label_set = set(label_chars)
    logprobs_by_label = {c: [] for c in label_chars}
    for t in top_logprobs:
        tok = t["token"].strip().upper()
        if tok in label_set:
            logprobs_by_label[tok].append(t["logprob"])

    combined = {}
    for c in label_chars:
        lps = logprobs_by_label[c]
        if lps:
            m = max(lps)
            combined[c] = m + np.log(np.sum(np.exp(np.array(lps) - m)))  # logsumexp
        else:
            combined[c] = FLOOR_LOGPROB
    return combined

In [17]:
def _first_answer_top_logprobs(content: list, label_chars: list):
    """Scan a logprobs.content list for the first token that resolves to a
    valid label and return its top_logprobs -- same scanning logic as
    model_calls.py's _first_label_probs, so reasoning-mode responses (where
    the answer isn't necessarily content[0]) are still handled correctly.
    Falls back to content[0] if nothing in the response matches.
    """
    if not content:
        return None
    label_set = set(label_chars)
    for entry in content:
        tok = entry["token"].strip().strip(".,:;)").upper()
        if tok in label_set:
            return entry["top_logprobs"]
    return content[0]["top_logprobs"]

In [18]:
def _infer_run_shape(npz_path: str):
    """(Q, J, K) for the run that produced npz_path, read off its
    distributions array shape (Q, J, K+1, C). Uses whichever seed key
    happens to be first -- Q/J/K are experiment config, constant across
    seeds stored in the same npz.
    """
    raw = np.load(npz_path, allow_pickle=True)
    seed = list(raw.keys())[0]
    distributions = np.asarray(raw[seed].item()["distributions"])
    Q, J, Kp1 = distributions.shape[0], distributions.shape[1], distributions.shape[2]
    return Q, J, Kp1 - 1


def parse_log_file(jsonl_path: str, Q: int = None, J: int = None, K: int = None) -> list:
    """Parse one raw_direct_query_responses_*.jsonl file into row dicts:
    prompt, finish_reason, resulting_probs (plus one column per class,
    resulting_prob_A, resulting_prob_B, ...), resulting_log_probs (the same
    per-class combined log-probabilities as a list, mirroring
    resulting_probs), and one combined log-probability column per class
    label (log_prob_A, log_prob_B, ...). The number of classes is read
    per-record from len(resulting_probs), so files that mix 4-choice and
    5-choice questions still parse correctly.

    If Q (questions), J (trajectories), K (self-conditioning steps) are
    given, also adds q/j/k columns identifying which (question, trajectory,
    step) state each row came from. This relies on run_martingale_check's
    fixed logging order -- for j in range(J): for k in range(K+1): for each
    question in order -- so state is recovered purely from row position;
    verified to exactly reproduce the corresponding npz's distributions
    array. Only applied when len(rows) == Q*J*(K+1) exactly, since a
    partial/stale log file (e.g. a crashed rerun) would silently mislabel
    every row after the point it went out of sync.
    """
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            resulting_probs = rec.get("resulting_probs")
            n_classes = len(resulting_probs) if resulting_probs else 0
            label_chars = [chr(ord('A') + i) for i in range(n_classes)]

            content = (
                rec.get("raw_response", {})
                   .get("choices", [{}])[0]
                   .get("logprobs", {})
                   .get("content")
            )

            row = {
                "prompt": rec.get("prompt"),
                "finish_reason": rec.get("finish_reason"),
                "resulting_probs": resulting_probs,
            }
            for i, c in enumerate(label_chars):
                row[f"resulting_prob_{c}"] = resulting_probs[i] if resulting_probs else None

            if content and n_classes:
                top_logprobs = _first_answer_top_logprobs(content, label_chars)
                combined = _label_logprobs_from_top_logprobs(top_logprobs, label_chars)
            else:
                combined = {c: FLOOR_LOGPROB for c in label_chars}
            row["resulting_log_probs"] = [combined[c] for c in label_chars]
            for c in label_chars:
                row[f"log_prob_{c}"] = combined[c]

            rows.append(row)

    if Q is not None and J is not None and K is not None and len(rows) == Q * J * (K + 1):
        for idx, row in enumerate(rows):
            j, rem = divmod(idx, Q * (K + 1))
            k, q = divmod(rem, Q)
            row["q"], row["j"], row["k"] = q, j, k
    elif Q is not None:
        print(f"[parse_log_file] WARNING: {osp.basename(jsonl_path)} has {len(rows)} rows, "
              f"expected Q*J*(K+1)={Q * J * (K + 1)} -- leaving q/j/k unset for this file.")

    return rows

In [19]:
def parse_log_folder(folder: str, pattern: str = LOG_FILE_PATTERN, npz_filename: str = NPZ_FILENAME) -> pd.DataFrame:
    """Parse every matching jsonl log file directly under `folder` into one
    DataFrame. `folder` is a run's .../martingale_check/<run_type> directory,
    e.g. RESULTS_ROOT/<dataset>_<model>/martingale_check/iterative.

    If `folder` contains a martingale_results.npz (it should, alongside the
    logs), (Q, J, K) is inferred from it once and used to add q/j/k columns
    to every row (see parse_log_file). Without it, rows just won't have
    q/j/k. Rows from questions with fewer classes than others found in the
    same folder get NaN in the extra log_prob_*/resulting_prob_* columns.
    """
    npz_path = osp.join(folder, npz_filename)
    Q = J = K = None
    if osp.exists(npz_path):
        try:
            Q, J, K = _infer_run_shape(npz_path)
        except Exception as e:
            print(f"[parse_log_folder] WARNING: failed to infer (Q, J, K) from {npz_path}: {e}")

    all_rows = []
    for path in sorted(glob.glob(osp.join(folder, pattern))):
        all_rows.extend(parse_log_file(path, Q=Q, J=J, K=K))
    return pd.DataFrame(all_rows)

In [20]:
DATASET = "obqa"
MODEL_NAME = "gpt4o_mini"
RUN_TYPE = "sampling"

log_folder = osp.join(RESULTS_ROOT, f"{DATASET}_{MODEL_NAME}", "martingale_check", RUN_TYPE)
df = parse_log_folder(log_folder)
print(f"parsed {len(df)} rows from {log_folder}")
df.head()

[parse_log_file] WARNING: raw_direct_query_responses_0904_1707_36.jsonl has 1962 rows, expected Q*J*(K+1)=500 -- leaving q/j/k unset for this file.
parsed 1962 rows from /Users/alicansahin/Desktop/LMU/SoSe 26/thesis/uq_with_martingale_posteriors/data/results_mp_sampling/obqa_gpt4o_mini/martingale_check/sampling


,prompt,finish_reason,resulting_probs,resulting_prob_A,resulting_prob_B,resulting_prob_C,resulting_prob_D,resulting_log_probs,log_prob_A,log_prob_B,log_prob_C,log_prob_D
0,Return the label of the correct answer for the...,stop,"[4.363462773051567e-09, 0.999999925775267, 1.6...",4.363463e-09,1.000000,1.605228e-09,6.825604e-08,"[-19.25, -1.9342090862297273e-07, -20.25, -16.5]",-19.250000,-1.934209e-07,-20.250000,-16.500000
1,Return the label of the correct answer for the...,stop,"[8.930360169789222e-06, 0.0003351092930868679,...",8.930360e-06,0.000335,7.094264e-04,9.989465e-01,"[-11.626053810119629, -8.001053810119629, -7.2...",-11.626054,-8.001054e+00,-7.251054,-0.001054
2,Return the label of the correct answer for the...,stop,"[6.691586875233685e-10, 0.9999998045542327, 9....",6.691587e-10,1.000000,9.237451e-09,1.855392e-07,"[-21.125, -3.1259903154892273e-07, -18.5, -15.5]",-21.125000,-3.125990e-07,-18.500000,-15.500000
3,Return the label of the correct answer for the...,stop,"[7.338033298733826e-07, 0.9999752600946338, 2....",7.338033e-07,0.999975,2.561225e-06,2.144488e-05,"[-14.125024795532227, -2.4748556811893764e-05,...",-14.125025,-2.474856e-05,-12.875025,-10.750025
4,Return the label of the correct answer for the...,stop,"[4.1399376856041394e-08, 0.9999995602114753, 5...",4.139938e-08,1.000000,5.602796e-09,3.927864e-07,"[-17.0, -4.3177378361572685e-07, -19.0, -14.75]",-17.000000,-4.317738e-07,-19.000000,-14.750000


In [21]:
excel_path = osp.join(log_folder, f"{DATASET}_{MODEL_NAME}_parsed_logs.xlsx")
df.to_excel(excel_path, index=False)
print(f"saved {len(df)} rows to {excel_path}")

saved 1962 rows to /Users/alicansahin/Desktop/LMU/SoSe 26/thesis/uq_with_martingale_posteriors/data/results_mp_sampling/obqa_gpt4o_mini/martingale_check/sampling/obqa_gpt4o_mini_parsed_logs.xlsx
